In [ ]:
from langchain_openai import ChatOpenAI  #测试api是否能打印
from dotenv import load_dotenv
import os
load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("ZHIPU_API_KEY"),
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    model="glm-4-flash"
)
print(llm.invoke("你好"))


上面是测试是否能连接畅通的代码，下面是具体进行调用的代码

In [ ]:
from langchain.agents import create_agent  #使用智能体模块
from langchain_openai import ChatOpenAI  #兼容接口
from langchain_core.tools import tool  #使用工具模块
from dotenv import load_dotenv  #导入api密钥
import os  #使用输入输出

load_dotenv()  #导入API接口
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")

# 工具定义
@tool  #使用工具装饰器
def get_weather(city: str) -> str:
    """获取指定城市的天气。
    Args:
        city: 需要查询天气的城市名称
    """
    return f"{city} 总是阳光明媚！"

# 智谱模型，兼容OpenAI接口
llm = ChatOpenAI(
    api_key=ZHIPU_API_KEY,
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    model="glm-4-flash",
    temperature=0.1  #使得回答更加精准尖锐
)

# create_agent 新版标准写法
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="你是乐于助人的助手，需要查询天气的时候调用get_weather工具"
)

# 调用
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "旧金山的天气怎么样？"}
    ]
})

# 打印最终回复
print(result["messages"][-1].content)




使用详细的系统 Prompt 改进智能体行为，在这里还没有使用工具，豆包的出错比较多，按照langchain谱写的入门教程一步一步来

In [ ]:
from langchain.agents import create_agent  #使用智能体模块
from langchain_openai import ChatOpenAI  #兼容接口
from dotenv import load_dotenv  #导入api密钥
import os  #使用输入输出

load_dotenv()  #导入API接口
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")

# 智谱模型，兼容OpenAI接口
llm = ChatOpenAI(
    api_key=ZHIPU_API_KEY,
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    model="glm-4-flash",
    temperature=0.1  #使得回答更加精准尖锐
)

# create_agent 新版标准写法
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt='''你是乐于助人的助手'''

)
# 调用
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "旧金山的天气怎么样？"}
    ]
})
# 打印最终回复
print(result["messages"][-1].content)

1、使用详细的系统 Prompt 改进智能体行为。
2、创建工具，与外部数据集成。
3、通过模型配置获得一致的响应。
4、使用对话记忆实现类似聊天的交互。
5、使用 Deep Agents 获得内置功能。
6、测试智能体。

In [5]:
import urllib.error    #给定url加载对应文档
import urllib.request  #url请求
from dotenv import load_dotenv  #加载对应的api-key
from langchain.agents import create_agent  #创建Agent
from deepagents import create_deep_agent  #创建deepAgent
from langchain.chat_models import init_chat_model  #使用适合当前用例的参数设置设置语言模型
from langchain.tools import tool  #使用工具
from langgraph.checkpoint.memory import InMemorySaver  #记忆模块

#加载API_KEY
load_dotenv()
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")

#PROMPT部分的内容
SYSTEM_PROMPT = """你是一名文学数据助手。

## 能力

- `fetch_text_from_url`：从 URL 加载文档文本并加入对话。
不要猜测行数或位置；应以保存文件的工具结果为依据。"""


@tool    #工具装饰器
def fetch_text_from_url(url: str) -> str:
    """从 URL 获取文档。
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"获取失败：{e}"
    text = raw.decode("utf-8", errors="replace")
    return text

#指定模型，并且调整一些模型参数
model = init_chat_model(
    "glm-4.6",
    model_provider="openai",
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    api_key=ZHIPU_API_KEY,
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

#记忆模块
checkpointer = InMemorySaver()

#创建Agent
agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

#创建deep——agent
deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)


#发出的对应prompt的详细内容
content = f"""古登堡计划提供了 F. Scott Fitzgerald《了不起的盖茨比》的完整纯文本版本。
URL: https://www.gutenberg.org/files/64317/64317-0.txt

请尽可能回答以下问题：

1) 完整的古登堡文件中有多少行包含子字符串 `Gatsby`（统计行数，而不是同一行内的出现次数；每行以换行符结束）。
2) 文件中第一处包含 `Daisy` 的行号（从 1 开始）。
3) 用两句话给出中立的内容梗概。

请尽力完成第 1 和第 2 项。如果在任何时刻发现无法使用现有工具和推理能力**验证**准确答案，
不要编造数字：将该字段设为 `null`，并在 `how_you_computed_counts` 中明确说明限制。
如果遇到任何错误，请报告错误类型和错误消息。"""

print("正在运行 create_agent...", flush=True)
#运行Agent
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
print("正在运行 create_deep_agent...", flush=True)
#运行deep_agent
deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)
print("\ncreate_agent:")
print(agent_result["messages"][-1].content_blocks)
print("\ncreate_deep_agent:")
print(deep_agent_result["messages"][-1].content_blocks)

正在运行 create_agent...
正在运行 create_deep_agent...

create_agent:
[{'type': 'text', 'text': '所有从古登堡计划获取文本的尝试均因 **SSL 证书验证失败**（`[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain`）而无法完成。因此我无法加载原文并逐行验证统计结果。\n\n---\n\n### 结果\n\n| # | 问题 | 答案 |\n|---|------|------|\n| 1 | 包含子字符串 `Gatsby` 的行数 | `null` |\n| 2 | 第一处包含 `Daisy` 的行号（从 1 开始） | `null` |\n| 3 | 中立内容梗概 | 见下 |\n\n---\n\n#### 1 & 2 — `how_you_computed_counts`\n\n无法计算。尝试通过以下 URL 拉取全文均失败（SSL 证书验证错误）：\n\n- `https://www.gutenberg.org/files/64317/64317-0.txt`\n- `https://www.gutenberg.org/cache/epub/64317/pg64317.txt`\n- `http://www.gutenberg.org/files/64317/64317-0.txt`\n- `https://gutenberg.org/files/64317/64317-0.txt`\n- `https://www.gutenberg.org/files/64317/64317.txt`\n\n错误类型：`urlopen error` — `[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1006)`\n\n由于无法获取原文，无法逐行扫描并统计 `Gatsby` 出现的行数或定位 `Daisy` 首次出现的行号。按照要求，不编造数字，将两项均设为 

In [2]:
from langchain.agents import create_agent
from dotenv import load_dotenv  #加载对应的api-key
import os
from langchain.chat_models import init_chat_model  #使用适合当前用例的参数设置设置语言模型
from langchain_core.utils.uuid import uuid7  #设置相同id
from langgraph.checkpoint.memory import InMemorySaver  #使用记忆模块
#加载API_KEY
load_dotenv()
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

model = init_chat_model(
    "glm-4.6",
    model_provider="openai",
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    api_key=ZHIPU_API_KEY,
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

agent = create_agent(
    model=model,
    tools=[get_weather],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant",
)

config = {"configurable": {"thread_id": str(uuid7())}}

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]},
    config=config,
)

# 同一对话中的后续轮次：复用相同的 thread_id 以保留历史记录
result = agent.invoke(
    {"messages": [{"role": "user", "content": "那明天呢？"}]},
    config=config,
)

上面的代码不仅仅使用了记忆模块，还用了统一的user——id使得大模型能够辨别是同一个人，使用记忆模块使得大模型能接上上一次的记忆，达到能够连续询问的效果，追踪user与大模型的调用时，我使用了LangSmith，它可以记录所有的对话过程，并且给出对应的建议